# TSA_ch13_ising

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/TSA/blob/main/TSA_ch13/TSA_ch13_ising/TSA_ch13_ising.ipynb)

Ising model simulation showing three market regimes via Metropolis algorithm: ordered (bubble), critical point (regime break), and disordered (normal market). Uses vectorized checkerboard updates on a 60x60 lattice for physically accurate equilibrium states.

**Course:** Time Series Analysis and Forecasting (TSA)  
**Author:** Daniel Traian Pele  
**Chapter:** 13 — LPPL Models for Bubble Detection

In [ ]:
!pip install yfinance scipy statsmodels matplotlib numpy pandas -q

In [ ]:
"""
LPPL Chapter 13 Charts — TSA Color Scheme + Real Data
- Transparent backgrounds everywhere
- Legends at the bottom (outside the plot)
- TSA color palette
- Real market data via yfinance + LPPL fitting via scipy
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import yfinance as yf
from scipy.optimize import differential_evolution
import json, os
import warnings
warnings.filterwarnings('ignore')

# ── TSA color scheme ──
MainBlue   = '#1A3A6E'
Crimson    = '#DC3545'
Forest     = '#2E7D32'
Amber      = '#B5853F'
Orange     = '#E67E22'
Purple     = '#8E44AD'
DarkGray   = '#333333'
MediumGray = '#808080'

# ── Global style ──
plt.rcParams.update({
    'figure.facecolor':   'none',
    'axes.facecolor':     'none',
    'savefig.facecolor':  'none',
    'legend.facecolor':   'none',
    'legend.edgecolor':   'none',
    'legend.framealpha':  0,
    'font.size':          12,
    'axes.titlesize':     14,
    'axes.labelsize':     12,
    'legend.fontsize':    11,
    'xtick.labelsize':    11,
    'ytick.labelsize':    11,
})

OUTPUT_DIR = '.'


def save_fig(fig, name, dpi=150):
    path = f'{OUTPUT_DIR}/ch13_lppl_{name}.png'
    fig.savefig(path, dpi=dpi, bbox_inches='tight', transparent=True, pad_inches=0.1)
    plt.close(fig)
    print(f'  Saved {path}')


# =========================================================================
# REAL DATA INFRASTRUCTURE
# =========================================================================
_data_cache = {}
ALL_PARAMS = {}

def download_prices(ticker, start, end):
    key = f"{ticker}_{start}_{end}"
    if key not in _data_cache:
        df = yf.download(ticker, start=start, end=end, progress=False)
        close = df['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        _data_cache[key] = close.dropna()
    return _data_cache[key]


def _lppl_linear(t, y, tc, m, omega):
    dt = tc - t
    ok = dt > 0
    t_v, y_v, dt_v = t[ok], y[ok], dt[ok]
    if len(t_v) < 10:
        return None
    f = dt_v ** m
    X = np.column_stack([np.ones(len(t_v)), f,
                         f * np.cos(omega * np.log(dt_v)),
                         f * np.sin(omega * np.log(dt_v))])
    coeffs, _, _, _ = np.linalg.lstsq(X, y_v, rcond=None)
    fitted = X @ coeffs
    ssr = float(np.sum((y_v - fitted) ** 2))
    return coeffs, ssr, fitted


def fit_lppl(prices, tc_range=None, m_range=(0.1, 0.9), omega_range=(4, 25)):
    y = np.log(prices.values.astype(float))
    t = np.arange(len(y), dtype=float)
    if tc_range is None:
        tc_range = (len(y) - 5, len(y) + len(y) * 0.2)

    def objective(p):
        tc, m, omega = p
        res = _lppl_linear(t, y, tc, m, omega)
        if res is None:
            return 1e12
        coeffs, ssr, _ = res
        if coeffs[1] >= 0:
            return ssr * 100
        return ssr

    result = differential_evolution(objective, [tc_range, m_range, omega_range],
                                     seed=42, maxiter=1000, tol=1e-12,
                                     popsize=40, mutation=(0.5, 1.5), recombination=0.9)
    tc, m, omega = result.x
    coeffs, ssr, _ = _lppl_linear(t, y, tc, m, omega)
    A, B, C1, C2 = coeffs
    C = np.sqrt(C1**2 + C2**2)
    phi = np.arctan2(C2, C1)
    lam = np.exp(2 * np.pi / omega)
    sst = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ssr / sst if sst > 0 else 0.0
    dt_all = np.maximum(tc - t, 0.01)
    f_all = dt_all ** m
    fitted_all = A + B*f_all + C1*f_all*np.cos(omega*np.log(dt_all)) + C2*f_all*np.sin(omega*np.log(dt_all))
    return {'tc': tc, 'm': m, 'omega': omega, 'A': A, 'B': B,
            'C': C, 'C1': C1, 'C2': C2, 'phi': phi,
            'lambda': lam, 'R2': r2, 'ssr': ssr,
            't': t, 'log_price': y, 'fitted_log': fitted_all, 'prices': prices}


def lppl_curve(t_arr, p):
    dt = np.maximum(p['tc'] - t_arr, 1e-6)
    f = dt ** p['m']
    return p['A'] + p['B']*f + p['C1']*f*np.cos(p['omega']*np.log(dt)) + p['C2']*f*np.sin(p['omega']*np.log(dt))


def bootstrap_ci(prices, p0, n_boot=200):
    y = np.log(prices.values.astype(float))
    t = np.arange(len(y), dtype=float)
    resid = y - p0['fitted_log']
    tc_b, m_b, w_b = [], [], []
    for i in range(n_boot):
        rng = np.random.RandomState(i)
        y_b = p0['fitted_log'] + rng.choice(resid, len(resid), replace=True)
        def obj(p):
            res = _lppl_linear(t, y_b, p[0], p[1], p[2])
            if res is None: return 1e12
            if res[0][1] >= 0: return res[1] * 100
            return res[1]
        try:
            r = differential_evolution(obj,
                [(max(p0['tc']-30, len(y)-5), p0['tc']+30),
                 (max(0.1, p0['m']-0.15), min(0.9, p0['m']+0.15)),
                 (max(4, p0['omega']-3), min(25, p0['omega']+3))],
                seed=i, maxiter=200, tol=1e-8, popsize=15)
            tc_b.append(r.x[0]); m_b.append(r.x[1]); w_b.append(r.x[2])
        except Exception:
            pass
    def ci(a):
        a = np.array(a)
        return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) > 10 else (np.nan, np.nan)
    return {'tc_ci': ci(tc_b), 'm_ci': ci(m_b), 'omega_ci': ci(w_b)}


# =========================================================================
# Metropolis Algorithm — Vectorized Checkerboard Updates
# =========================================================================
def _checkerboard_sweep(spins, beta, parity):
    n = spins.shape[0]
    rows, cols = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
    mask = (rows + cols) % 2 == parity
    nb = (np.roll(spins, 1, 0) + np.roll(spins, -1, 0) +
          np.roll(spins, 1, 1) + np.roll(spins, -1, 1))
    dE = 2.0 * spins * nb
    prob = np.exp(-beta * dE)
    flip = (dE <= 0) | (np.random.random((n, n)) < prob)
    spins[mask & flip] *= -1
    return spins


def _equilibrate(spins, T, sweeps=3000):
    if T < 0.01:
        return spins
    beta = 1.0 / T
    for _ in range(sweeps):
        _checkerboard_sweep(spins, beta, 0)
        _checkerboard_sweep(spins, beta, 1)
    return spins


# =========================================================================
# TSA_ch13_ising
# =========================================================================
def chart_ising():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    N = 60
    Tc = 2.0 / np.log(1.0 + np.sqrt(2.0))
    cmap = LinearSegmentedColormap.from_list('ising', [Crimson, 'white', MainBlue])

    # Low T — ordered (bubble regime)
    np.random.seed(42)
    ax = axes[0]
    low_T = np.ones((N, N), dtype=int)
    low_T = _equilibrate(low_T, 0.5 * Tc)
    ax.imshow(low_T, cmap=cmap, vmin=-1, vmax=1)
    ax.set_title('$T < T_c$: Ordered\n$|m| \\approx 1$ (Strong consensus)', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, -0.12, 'Market: Strong herding\nBubble regime',
            transform=ax.transAxes, ha='center', fontsize=11, style='italic')

    # Critical
    np.random.seed(42)
    ax = axes[1]
    crit = np.ones((N, N), dtype=int)
    crit = _equilibrate(crit, Tc)
    ax.imshow(crit, cmap=cmap, vmin=-1, vmax=1)
    ax.set_title('$T = T_c$: Critical Point\nClusters of ALL sizes!', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, -0.12, 'Market: Maximum instability\nSmall trigger $\\to$ large cascade',
            transform=ax.transAxes, ha='center', fontsize=11, style='italic', color=Crimson)

    # High T — disordered (normal market)
    np.random.seed(42)
    ax = axes[2]
    high_T = np.ones((N, N), dtype=int)
    high_T = _equilibrate(high_T, 2.0 * Tc)
    ax.imshow(high_T, cmap=cmap, vmin=-1, vmax=1)
    ax.set_title('$T > T_c$: Disordered\n$m \\approx 0$ (No consensus)', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, -0.12, 'Market: Random trading\nNo herding behavior',
            transform=ax.transAxes, ha='center', fontsize=11, style='italic')

    handles = [
        mpatches.Patch(color=MainBlue, label='Spin Up (+1) = BUY'),
        mpatches.Patch(color=Crimson,  label='Spin Down (-1) = SELL'),
        mpatches.Patch(color='white', edgecolor='gray', label='Neutral'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=3,
               bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=12)
    plt.tight_layout(); plt.subplots_adjust(bottom=0.18)
    save_fig(fig, 'ising')

In [ ]:
chart_ising()

## Result

![TSA_ch13_ising](ch13_lppl_ising.png)

## Quiz

**Q1.** In the 2D Ising model, what happens to the magnetization $|M|$ at the critical temperature $T_c$?

**(A)** $|M|$ remains at 1 (perfect order persists)  
**(B)** $|M|$ drops continuously to 0 (second-order phase transition)  
**(C)** $|M|$ jumps discontinuously from 1 to 0 (first-order transition)  
**(D)** $|M|$ oscillates between 0 and 1

---

**Q2.** In the financial analogy of the Ising model, what does the regime $T < T_c$ (ordered phase) correspond to?

**(A)** Efficient market with independent, random trading  
**(B)** Strong herding — bubble regime with one group dominating  
**(C)** High-frequency trading with no directional bias  
**(D)** Bear market with declining prices

---

**Q3.** Why does the susceptibility $\chi \to \infty$ at the critical point $T_c$ matter for financial markets?

**(A)** It means the market becomes insensitive to news  
**(B)** It means the market is perfectly efficient  
**(C)** It means even a small perturbation can trigger a system-wide cascade  
**(D)** It means volatility drops to zero

### Answers

**Q1: (B)** The 2D Ising model undergoes a continuous (second-order) phase transition at $T_c = 2/\ln(1+\sqrt{2}) \approx 2.269$. The magnetization vanishes as $|M| \sim (T_c - T)^{\beta}$ with $\beta = 1/8$ (Onsager, 1944). There is no discontinuous jump.

**Q2: (B)** Below $T_c$, spins align — most agents make the same decision (buy or sell). This represents herding behavior and bubble formation: one group dominates, prices deviate from fundamentals. Above $T_c$, spins are random — agents trade independently, consistent with the Efficient Market Hypothesis.

**Q3: (C)** Diverging susceptibility means the system's response to a perturbation becomes infinite. In financial terms: near the critical point, a minor piece of news (a small "external field") can flip large clusters of traders simultaneously, producing a market-wide cascade. This explains why bubbles are fragile — the system is maximally sensitive just before a regime break.